# 01 · Byte-Pair Encoding (BPE) tokenizer from scratch

**Track:** Training LLMs & Frontier Research · **Runtime (CPU default):** ~10 seconds · **GPU:** not needed

Before a language model sees a single gradient, someone has decided what a "token" is. That choice quietly shapes
everything downstream: how many tokens a sentence costs, why GPT models used to be bad at arithmetic and at reversing
strings, and why a Hebrew or Hindi sentence can cost 3–5× more tokens than the same sentence in English. In this
notebook we build the byte-level BPE algorithm used by GPT-2/3/4, Llama 3 and most modern LLMs: first the 15-line
core, then the regex pre-tokenization and special-token handling that production tokenizers add.

> **Source & license credit.** The algorithm structure (`get_stats` / `merge` / `train` / `encode` / `decode`) and the
> GPT-2 / GPT-4 split regexes follow **Andrej Karpathy's [minbpe](https://github.com/karpathy/minbpe) (MIT License)**,
> which in turn reproduces the patterns from **OpenAI's [gpt-2](https://github.com/openai/gpt-2) ("Modified MIT"
> license) and [tiktoken](https://github.com/openai/tiktoken) (MIT)**. The code here was rewritten for this notebook
> (with a chunk-frequency training speed-up and a vocab-size sweep). The GPT-2 reference tokenizer
> ([openai-community/gpt2](https://huggingface.co/openai-community/gpt2), MIT per its model card) is downloaded at
> runtime via Hugging Face [`transformers`](https://github.com/huggingface/transformers) (Apache-2.0) and is not
> redistributed. The training text is the first ~300 KB of the `tinyshakespeare` file from Karpathy's
> [char-rnn](https://github.com/karpathy/char-rnn/tree/master/data/tinyshakespeare) (its README says "License: MIT");
> the text itself is Shakespeare and in the public domain. Full list and provenance notes:
> [`research/THIRD_PARTY_NOTICES.md`](../THIRD_PARTY_NOTICES.md).

## Learning goals

By the end you should be able to:

1. Explain why modern LLMs tokenize **bytes**, not characters or words.
2. Implement BPE training (iteratively merge the most frequent adjacent pair) and BPE encoding/decoding.
3. Explain what regex **pre-tokenization** buys you, and implement it.
4. Handle **special tokens** like `<|endoftext|>` safely.
5. Measure the **compression ratio vs. vocabulary size** trade-off, and see how a tokenizer trained on one language
   treats another.

## Background

* **BPE as compression** — Philip Gage, *A New Algorithm for Data Compression*, C Users Journal, 1994.
* **BPE for NMT** — Sennrich, Haddow & Birch, *Neural Machine Translation of Rare Words with Subword Units*, ACL 2016.
  [arXiv:1508.07909](https://arxiv.org/abs/1508.07909)
* **Byte-level BPE** — Radford et al., *Language Models are Unsupervised Multitask Learners* (GPT-2), OpenAI 2019.
  [PDF](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)
* **SentencePiece / Unigram LM** (the main alternative) — Kudo & Richardson, 2018.
  [arXiv:1808.06226](https://arxiv.org/abs/1808.06226); Kudo, 2018. [arXiv:1804.10959](https://arxiv.org/abs/1804.10959)
* **Tokenization disparity across languages** — Petrov et al., *Language Model Tokenizers Introduce Unfairness Between
  Languages*, NeurIPS 2023. [arXiv:2305.15425](https://arxiv.org/abs/2305.15425)

**The idea in one paragraph.** Start with the 256 possible byte values as the vocabulary; any string is a sequence of
UTF-8 bytes, so nothing is ever out-of-vocabulary. Count every adjacent pair of tokens in the training corpus, mint a
new token for the most frequent pair, replace all its occurrences, and repeat until the vocabulary has the desired
size. Each merge shortens the corpus, so BPE is literally a greedy compression algorithm; the ordered list of merges
*is* the tokenizer.

In [ ]:
# --- Setup: make research/src importable (from research/, research/notebooks/, or a Colab clone) ---
import sys, pathlib
for _p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]:
    _src = next((c for c in (_p / "src", _p / "research" / "src") if (c / "lab_config.py").exists()), None)
    if _src:
        sys.path.insert(0, str(_src))
        break
else:
    raise RuntimeError("Could not find research/src -- run this notebook from inside the repo.")

from lab_config import CFG, pick, set_seed, heavy_cell, read_text, quiet_hf_logging

FULL_SIZE = CFG.full_size   # <- set True here (or export LAB_FULL_SIZE=1) for the full-size run
CFG.full_size = FULL_SIZE
set_seed()
print(CFG.summary())

In [ ]:
import json, time
from collections import Counter
import regex as re          # the `regex` package supports \p{L} / \p{N} Unicode classes
import matplotlib.pyplot as plt

text = read_text("tiny_shakespeare.txt")
print(f"{len(text):,} characters, {len(text.encode('utf-8')):,} bytes")
print(text[:250])

## 1 · Bytes are the alphabet

UTF-8 encodes ASCII in 1 byte, Hebrew/Greek/Cyrillic in 2, most CJK in 3 and emoji in 4. A byte-level model therefore
starts with a fixed 256-symbol alphabet that can represent *any* string.

In [ ]:
for s in ["hello", "שלום", "héllo", "你好", "🙂"]:
    b = s.encode("utf-8")
    print(f"{s!r:10} -> {len(s)} chars, {len(b)} bytes: {list(b)}")

## 2 · The core of BPE: count pairs, merge the most frequent

In [ ]:
def get_stats(ids, counts=None, weight=1):
    """Count adjacent pairs in `ids`. `weight` lets us count a chunk that occurs many times in one go."""
    counts = {} if counts is None else counts
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + weight
    return counts


def merge(ids, pair, idx):
    """Replace every occurrence of `pair` in `ids` with the new token `idx`."""
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            out.append(idx)
            i += 2
        else:
            out.append(ids[i])
            i += 1
    return out


ids = list("aaabdaaabac".encode("utf-8"))  # the toy example from the Wikipedia BPE article
stats = get_stats(ids)
top = max(stats, key=stats.get)
print("pair counts:", {(chr(a), chr(b)): c for (a, b), c in stats.items()})
print("most frequent:", (chr(top[0]), chr(top[1])), "-> new token 256")
print("after one merge:", merge(ids, top, 256))

### A minimal tokenizer (no pre-tokenization)

Training is just that loop, repeated `vocab_size - 256` times. Encoding replays the merges **in the order they were
learned** (lowest merge index first); decoding concatenates each token's bytes.

In [ ]:
class BasicTokenizer:
    def __init__(self):
        self.merges = {}                                  # (int, int) -> int, in insertion (= training) order
        self.vocab = {i: bytes([i]) for i in range(256)}  # int -> bytes

    def train(self, text, vocab_size, verbose=False):
        ids = list(text.encode("utf-8"))
        for i in range(vocab_size - 256):
            stats = get_stats(ids)
            if not stats:
                break
            pair = max(stats, key=stats.get)
            idx = 256 + i
            ids = merge(ids, pair, idx)
            self.merges[pair] = idx
            self.vocab[idx] = self.vocab[pair[0]] + self.vocab[pair[1]]
            if verbose and i < 10:
                print(f"merge {i + 1:3d}: {self.vocab[pair[0]]!r:>8} + {self.vocab[pair[1]]!r:<8} -> {self.vocab[idx]!r} ({stats[pair]} times)")
        return self

    def encode(self, text):
        ids = list(text.encode("utf-8"))
        while len(ids) >= 2:
            stats = get_stats(ids)
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))  # earliest-learned merge
            if pair not in self.merges:
                break
            ids = merge(ids, pair, self.merges[pair])
        return ids

    def decode(self, ids):
        return b"".join(self.vocab[i] for i in ids).decode("utf-8", errors="replace")


sample = text[:20_000]
t0 = time.time()
basic = BasicTokenizer().train(sample, vocab_size=256 + 64, verbose=True)
print(f"trained 64 merges on 20 KB in {time.time() - t0:.1f}s")

enc = basic.encode(sample)
assert basic.decode(enc) == sample, "round-trip failed!"
print(f"compression: {len(sample.encode()):,} bytes -> {len(enc):,} tokens ({len(sample.encode()) / len(enc):.2f}x)")

Notice what the basic tokenizer learns: tokens like `'e '` or `', '` that glue a word ending to the following space or
punctuation. That wastes vocabulary on many near-duplicates (`"dog."`, `"dog!"`, `"dog?"`) and lets merges cross word
boundaries. GPT-2 fixed this with a **pre-tokenization regex**.

## 3 · Regex pre-tokenization (GPT-2 / GPT-4 style)

The text is first split into chunks (roughly: words with their leading space, numbers, punctuation runs, whitespace);
BPE merges are then only allowed *inside* a chunk.

In [ ]:
# Patterns from openai/gpt-2 (encoder.py) and openai/tiktoken (cl100k_base), as collected in karpathy/minbpe. MIT.
GPT2_SPLIT_PATTERN = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
GPT4_SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

demo = "Hello world!! It's 2026, and I've got 12345 tokens...   OK?"
print("GPT-2:", re.findall(GPT2_SPLIT_PATTERN, demo))
print("GPT-4:", re.findall(GPT4_SPLIT_PATTERN, demo))

Two details worth noticing: GPT-4's pattern splits numbers into groups of **at most 3 digits** (so a long number is
never one rare token of its own; this is widely believed to help with arithmetic, although OpenAI has not documented
the motivation), and it is case-insensitive for contractions (`'S` as well as `'s`).

### Training trick: count each unique chunk once

The corpus has tens of thousands of chunks but far fewer *unique* chunks (printed below). Since merges never cross chunk boundaries, we can
run BPE over the unique chunks and weight each pair count by the chunk's frequency. Same result as the naive loop,
roughly an order of magnitude faster in pure Python.

In [ ]:
class RegexTokenizer(BasicTokenizer):
    def __init__(self, pattern=GPT4_SPLIT_PATTERN):
        super().__init__()
        self.pattern = pattern
        self.compiled = re.compile(pattern)
        self.special_tokens = {}       # str -> int
        self.inverse_special = {}      # int -> str
        self._cache = {}

    def train(self, text, vocab_size, verbose=False):
        chunk_freq = Counter(self.compiled.findall(text))
        chunks = [list(ch.encode("utf-8")) for ch in chunk_freq]
        freqs = list(chunk_freq.values())
        for i in range(vocab_size - 256):
            stats = {}
            for ch, f in zip(chunks, freqs):
                get_stats(ch, stats, weight=f)
            if not stats:
                break
            pair = max(stats, key=stats.get)
            idx = 256 + i
            chunks = [merge(ch, pair, idx) if len(ch) > 1 else ch for ch in chunks]
            self.merges[pair] = idx
            self.vocab[idx] = self.vocab[pair[0]] + self.vocab[pair[1]]
            if verbose and (i < 8 or (i + 1) % 128 == 0):
                print(f"merge {i + 1:4d}: {self.vocab[idx]!r} ({stats[pair]:,} occurrences)")
        self._cache = {}
        return self

    def register_special_tokens(self, special):
        self.special_tokens = dict(special)
        self.inverse_special = {v: k for k, v in special.items()}

    def _encode_chunk(self, chunk_bytes):
        if chunk_bytes in self._cache:
            return self._cache[chunk_bytes]
        ids = list(chunk_bytes)
        while len(ids) >= 2:
            stats = get_stats(ids)
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges:
                break
            ids = merge(ids, pair, self.merges[pair])
        self._cache[chunk_bytes] = ids
        return ids

    def encode_ordinary(self, text):
        """Encode ignoring special tokens (they are treated as plain text)."""
        out = []
        for chunk in self.compiled.findall(text):
            out.extend(self._encode_chunk(chunk.encode("utf-8")))
        return out

    def encode(self, text, allowed_special="none"):
        """allowed_special: "all" (opt in) | "none" (special strings become plain text) | "none_raise" (refuse them)."""
        if allowed_special not in ("all", "none", "none_raise"):
            raise ValueError(f"unknown allowed_special={allowed_special!r}")
        if allowed_special == "none" or not self.special_tokens:
            return self.encode_ordinary(text)
        if allowed_special == "none_raise":
            for s in self.special_tokens:
                if s in text:
                    raise ValueError(f"special token {s!r} found in text; pass allowed_special='all' if intended")
            return self.encode_ordinary(text)
        special_re = "(" + "|".join(re.escape(s) for s in self.special_tokens) + ")"
        out = []
        for part in re.split(special_re, text):
            if part in self.special_tokens:
                out.append(self.special_tokens[part])
            elif part:
                out.extend(self.encode_ordinary(part))
        return out

    def decode(self, ids):
        parts = [self.vocab[i] if i in self.vocab else self.inverse_special[i].encode("utf-8") for i in ids]
        return b"".join(parts).decode("utf-8", errors="replace")

    def save(self, path):
        data = {"pattern": self.pattern, "merges": [[a, b, idx] for (a, b), idx in self.merges.items()],
                "special_tokens": self.special_tokens}
        pathlib.Path(path).write_text(json.dumps(data))

    @classmethod
    def load(cls, path):
        data = json.loads(pathlib.Path(path).read_text())
        tok = cls(data["pattern"])
        for a, b, idx in data["merges"]:
            tok.merges[(a, b)] = idx
            tok.vocab[idx] = tok.vocab[a] + tok.vocab[b]
        tok.register_special_tokens(data["special_tokens"])
        return tok

In [ ]:
VOCAB_SIZE = pick(256 + 512, 256 + 4096)   # CPU default: 512 merges (~5-10 s). Full: 4096 merges.

n_chunks = len(re.findall(GPT4_SPLIT_PATTERN, text))
n_unique = len(set(re.findall(GPT4_SPLIT_PATTERN, text)))
print(f"{n_chunks:,} chunks, {n_unique:,} unique -> pair counting is ~{n_chunks / n_unique:.0f}x cheaper")

t0 = time.time()
tok = RegexTokenizer().train(text, VOCAB_SIZE, verbose=True)
print(f"\ntrained {len(tok.merges)} merges on {len(text):,} chars in {time.time() - t0:.1f}s")

In [ ]:
ids = tok.encode(text)
assert tok.decode(ids) == text
n_bytes = len(text.encode("utf-8"))
print(f"compression: {n_bytes:,} bytes -> {len(ids):,} tokens = {n_bytes / len(ids):.2f} bytes/token")

longest = sorted(tok.vocab.values(), key=len, reverse=True)[:15]
print("longest learned tokens:", [t.decode("utf-8", errors="replace") for t in longest])

sentence = "ROMEO: But soft, what light through yonder window breaks?"
print([tok.decode([i]) for i in tok.encode(sentence)])

## 4 · Special tokens

Special tokens such as `<|endoftext|>` separate documents during pre-training and mark chat-turn boundaries during
fine-tuning. They must **never** be produced by accident from user text — otherwise a user can inject a fake
"end of system prompt". tiktoken's default is therefore to *refuse* text containing special strings unless the caller
opts in. Our `encode` offers the same three behaviours through `allowed_special`: `"all"` (opt in), `"none_raise"`
(refuse, which is tiktoken's default) and `"none"` (encode the special string as ordinary text; our default, to keep
the demos short). In code that handles untrusted input, prefer `"none_raise"`.

In [ ]:
EOT = "<|endoftext|>"
tok.register_special_tokens({EOT: VOCAB_SIZE})
doc = f"First document.{EOT}Second document."

print("allowed  :", tok.encode(doc, allowed_special="all"))
print("as text  :", tok.encode(doc, allowed_special="none")[:12], "...")
try:
    tok.encode(doc, allowed_special="none_raise")
except ValueError as e:
    print("refused  :", e)

## 5 · Compression vs. vocabulary size

Because merges are learned greedily in order, the tokenizer with `k` merges is just a *prefix* of the merge list. We
can therefore measure many vocabulary sizes from one training run. Larger vocabularies compress more (shorter
sequences → more context per FLOP) but cost a bigger embedding matrix and rarer, less-trained tokens.

In [ ]:
def prefix_tokenizer(full, k):
    t = RegexTokenizer(full.pattern)
    for (pair, idx) in list(full.merges.items())[:k]:
        t.merges[pair] = idx
        t.vocab[idx] = full.vocab[idx]
    return t

held_out = text[-30_000:]          # note: inside the training text; see "Your turn" for a true held-out test
ks = [0, 16, 32, 64, 128, 256, len(tok.merges)]
ks = sorted(set(k for k in ks if k <= len(tok.merges)))
ratios = [len(held_out.encode()) / len(prefix_tokenizer(tok, k).encode(held_out)) for k in ks]
for k, r in zip(ks, ratios):
    print(f"{256 + k:5d} vocab -> {r:.2f} bytes/token")

plt.figure(figsize=(5, 3.2))
plt.plot([256 + k for k in ks], ratios, "o-")
plt.xscale("log", base=2)
plt.xlabel("vocabulary size"); plt.ylabel("bytes per token")
plt.title("BPE compression on Shakespeare"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 6 · Compare with GPT-2's tokenizer — and try another language

GPT-2's 50,257-token vocabulary was trained on ~40 GB of English web text (WebText). Ours was trained on 300 KB of
Shakespeare. Let's compare on English, on code, and on Hebrew (which neither tokenizer was meant for).

In [ ]:
quiet_hf_logging()
from transformers import AutoTokenizer
gpt2 = AutoTokenizer.from_pretrained("gpt2")   # tokenizer files only (~1–2 MB)

samples = {
    "Shakespeare": "O Romeo, Romeo! wherefore art thou Romeo? Deny thy father and refuse thy name.",
    "modern English": "The quarterly earnings report was released on the company's website yesterday.",
    "Python code": "def add(a, b):\n    return a + b  # 12345 + 678",
    "Hebrew": "שלום עולם, זהו מבחן קטן של טוקנייזר.",
}
print(f"{'text':16} {'bytes':>6} {'ours':>6} {'gpt2':>6}")
for name, s in samples.items():
    print(f"{name:16} {len(s.encode()):6d} {len(tok.encode(s)):6d} {len(gpt2.encode(s)):6d}")

print("\nGPT-2 pieces for the Hebrew text:", gpt2.convert_ids_to_tokens(gpt2.encode(samples["Hebrew"]))[:12], "...")

**What to notice** (check the table above; the numbers quoted here are from the default 512-merge run and change
with `VOCAB_SIZE`). GPT-2 wins everywhere, which is no surprise with a ~65× larger vocabulary trained on ~100,000×
more text. But the *gap* is smallest on Shakespeare, the distribution our tokenizer was trained on (31 vs. 20 tokens,
1.6×), and largest on modern prose (36 vs. 13, 2.8×), with code in between (38 vs. 22, 1.7×): a tokenizer is
specialised to its training corpus. Four one-line samples are an illustration, not a benchmark. On Hebrew our
tokenizer falls back to raw bytes (exactly one token per byte, i.e. two tokens per Hebrew letter) and GPT-2, at 44
tokens for 64 bytes, does only a little better. This is the "tokenization tax" Petrov et al. (2023) measure across
languages: speakers of under-represented languages pay more per request, get less effective context, and the model
sees fewer "words" of their language per training FLOP.

## 7 · Save and reload

In [ ]:
out_path = CFG.output_dir / "shakespeare_bpe.json"
tok.save(out_path)
tok2 = RegexTokenizer.load(out_path)
assert tok2.encode(sentence) == tok.encode(sentence)
print("saved and reloaded:", out_path.name, f"({out_path.stat().st_size / 1024:.1f} KB)")

## Your turn

1. **True held-out test.** Hold out the last 10% of the text *before* training and re-measure bytes/token. How much of
   the compression above was memorisation?
2. **GPT-2 vs GPT-4 regex.** Train two tokenizers with the two split patterns and compare how they tokenize
   `"In 1599, 2026 and 31415926"`. Which one would you rather do arithmetic with?
3. **Hebrew-aware vocabulary.** Append a short Hebrew paragraph (written by you) to the training text and retrain.
   How many merges does it take before common Hebrew letters become single tokens?
4. **Speed.** Profile `RegexTokenizer.train`. Where does the time go? Sketch how an incremental pair-count update
   (only touching chunks that contain the merged pair) would make each merge sub-linear.

## Beyond the paper — a mini research question

*Does a tokenizer trained on the pre-training corpus actually make a small model better, per byte?* Notebook 02 trains a
character-level GPT. Re-run it with this BPE tokenizer at vocab sizes 256 / 512 / 1024 and compare models by
**bits per byte** (not loss per token — tokens differ in size!). For a fixed compute budget, is there an optimal
vocabulary size for a ~1M-parameter model? (Related: Tao et al., *Scaling Laws with Vocabulary*, 2024,
[arXiv:2407.13623](https://arxiv.org/abs/2407.13623).)

## References

* Gage, P. (1994). *A New Algorithm for Data Compression.* C Users Journal.
* Sennrich, R., Haddow, B., Birch, A. (2016). *Neural Machine Translation of Rare Words with Subword Units.* [arXiv:1508.07909](https://arxiv.org/abs/1508.07909)
* Radford, A. et al. (2019). *Language Models are Unsupervised Multitask Learners.* OpenAI. [PDF](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)
* Kudo, T., Richardson, J. (2018). *SentencePiece.* [arXiv:1808.06226](https://arxiv.org/abs/1808.06226)
* Petrov, A. et al. (2023). *Language Model Tokenizers Introduce Unfairness Between Languages.* [arXiv:2305.15425](https://arxiv.org/abs/2305.15425)
* Tao, C. et al. (2024). *Scaling Laws with Vocabulary.* [arXiv:2407.13623](https://arxiv.org/abs/2407.13623)
* Karpathy, A. *minbpe* (MIT) — https://github.com/karpathy/minbpe · OpenAI *tiktoken* (MIT) — https://github.com/openai/tiktoken